In [ ]:
# -*- coding: utf-8 -*-
#  Copyright 2024 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#  Authored by:    Rasmia Kulan (UKRI-STFC)
#                  Laura Murgatroyd (UKRI-STFC)

# Introduction to Aluminium Cylinder Flexible Neutron Tomography Dataset

Learning objectives:
- Understand the sample and acquisitions that were made.
- Load the data, which has been pre-processed in Mantid Imaging
- Reconstruct, using Filtered Backprojection, 3 slices, which we will use throughout the demos with this dataset.


In [ ]:
from cil.utilities.display import show2D
import numpy as np
import matplotlib.pyplot as plt

## Data

The dataset used in the Angles_vs_Exposure notebooks is the Aluminium Cylinder Flexible Neutron Tomography Dataset.
This is available on zenodo at: https://zenodo.org/records/17250237

Specifically, these notebooks work with the pre-processed data, for which the direct download link is: https://zenodo.org/records/17250237/files/preprocessed_data.zip?download=1



### The Sample

There is an upper and lower cylinder in this sample, both made of Aluminium.

The upper cylinder contains four rods, each 2mm in diameter. The material was unknown but a portable element analyzer indicated that the rods are made of ~90% Si alloyed with small amounts of Fe/Al.  

The rods also extend out of the top of the cylinder.

The lower cylinder has holes with 1mm, 2mm, 3mm and 4mm diameters, and they are filled with:

·       ZrO2 spheres (0.5mm diameter),

·       Trisospheres (1mm diameter),

·       Mixture of ZrO2 (0.5mm diameter) and trisopheres (1mm diameter),

·       Steel spheres (2mm dia) in Al foil

The trisospheres have a ZrO2 kernel directly coated with a pyrolytic carbon layer and an outer SiC layer. The SiC layer is expected to have been too thin to be resolved in the experiment.


![al_cyl_diagram](images/al_cyl_diagram.png "Aluminium Cylinder Diagram")

## The Acquisition
The cylinder was scanned at IMAT, ISIS Neutron & Muon Source, experiment number RB2480001.
The data was acquired at multiple different exposure times, and with a highly composite number of angles, allowing investigations of optimal ratio between angles and exposure times when total experiment time is limited.
 
The following exposure times were acquired:

30s

15s

7.5s

3.75s - two acquisitions were made at this exposure time, labelled 'a' and 'b'

These times were chosen as they add up to give 1 minute.

Each of these acquisitions inlcude 840 projections, which is above the Nyquist criterion. The pixel size was 48 microns.

## Load the Pre-processed data

We have already created pre-processed datasets, which will be used in these notebooks!
Once you have downloaded the data from zenodo, you need to set the `processed_data_path` variable in [data_io/alum_cyl_file_paths.py](data_io/alum_cyl_file_paths.py) to be the path to that folder. This will mean all of the demo notebooks know where to find the data.

Then we'll load in preprocessed data with 60s exposure time and all of the acquired angles. This was achieved by summing all of the acquisitions we made:

In [ ]:
from data_io.alum_cyl_io import read_processed_data
data = read_processed_data(exposure_time=60, num_angles=840)

Here we show a single projection of the 60 second acquistion.

In [ ]:
show2D(data, title='Pre-processed 60s acquisition', cmap='inferno', origin='upper')

We have chosen three different Region of Interests from the cylinder as shown below:

<img src="images/cyl_slices.png" width=500 align="center">

In [ ]:
#Use CIL to slice the pre-processed data at these locations
from cil.processors import Slicer

sliceA_slicer = Slicer(roi={'vertical': (245, 246)})
sliceB_slicer = Slicer(roi={'vertical': (623, 624)})
sliceC_slicer = Slicer(roi={'vertical': (987, 988)})

exp_1min_slice1 = sliceA_slicer(data)
exp_1min_slice2 = sliceB_slicer(data)
exp_1min_slice3 = sliceC_slicer(data)

In [ ]:
show2D([exp_1min_slice1, exp_1min_slice2, exp_1min_slice3],['Slice A [vertical=245]', 'Slice B [vertical=623]', 'Slice C [vertical=987]'], num_cols=3, cmap='inferno', fix_range=(0, np.max(exp_1min_slice3)))

The three regions are chosen based on various parts of the sample:
- Slice A is cut through where there is just rods and air (no cylinder background)
- Slice B is a cut through where there are rods in the holes through the cylinder
- Slice C is a cut through the region where there are spheres in the holes through the cylinder

## Reconstruct using FBP

Here we import FBP from CIL's ASTRA plugin and use it to reconstruct the data:

In [ ]:
from cil.plugins.astra import FBP as FBP_ASTRA

data.reorder('astra')
recon = FBP_ASTRA(data.geometry.get_ImageGeometry(), data.geometry)(data)

exp_1min_rec1 = sliceA_slicer(recon)
exp_1min_rec2 = sliceB_slicer(recon)
exp_1min_rec3 = sliceC_slicer(recon)


all_rec = [
    exp_1min_rec1, exp_1min_rec2, exp_1min_rec3
]

all_labels = [
'Slice A FBP [vertical=445]','Slice B FBP [vertical=823, angle=840]', 'Slice C FBP [vertical=1187, angle=840]'
]

show2D(all_rec,all_labels, cmap='inferno', num_cols=3, fix_range=(0, np.max(exp_1min_rec3)))


Here we plot histograms of the reconstructed slices:

In [ ]:
def show_hist_for_recons(dataset,start, stop, label1, label2):
    # Convert dataset to 2D array
    data_2d = dataset.as_array()

    # Flatten the 2D array to 1D
    data = data_2d.flatten()
    
    # Create a new figure with subplots
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    
    # Plot first histogram without range constraint
    axs[0].hist(data, bins=1000, density=True, alpha=0.6, label=f'Histogram {label1}')
    axs[0].set_xlabel('Pixel Values')
    axs[0].set_ylabel('Number of Pixels')
    axs[0].set_title(f'{label1}')
    
    # Plot histogram with range of interest
    axs[1].hist(data, bins=1000, density=True, alpha=0.6, range=(start, stop), label=f'Histogram {label2}')
    axs[1].set_xlabel('Pixel Values')
    axs[1].set_ylabel('Number of Pixels')
    axs[1].set_title(f'{label2}')

    # Adjust layout
    plt.tight_layout()
    
    # Display the histograms
    plt.show()


show_hist_for_recons(exp_1min_rec1,0.25,2, 'Full Slice A', 'Cropped Slice A')
show_hist_for_recons(exp_1min_rec2,0.25,2, 'Full Slice B','Cropped Slice B')
show_hist_for_recons(exp_1min_rec3,0.25,2,'Full Slice C', 'Cropped Slice C')


- Slice A - shows a large peak at ~0 as it is mostly air, along with peaks at ~0.9 and ~1.6 indicating the contrast of the two types of rods.

- Slice B - Two large peaks at ~0 and ~0.1 from the air background and the aluminium cylinder. Plus again we see peaks indicating the contrast of the two types of rods.

- Slice C - Again we see peaks for the air and aluminium cylinder. We see less defined peaks at higher pixel values, as the ROI within the cylinder are very small and the foil and glue around the cylinder has a broader range of pixel values.